# 7. Test split: on its own, and against everything else (CPU, no download)

Reads saved results only. First every table for the TEST scenes alone. Then the comparison that is fair: test
against all other scenes **inside the same weather and light**, because the test split is all daytime and the other
scenes include the night drives. `intervals_overlap = True` means the two are not clearly different.

In [1]:
# --- 1. Configuration ---
PERSIST_MODE = "drive"
DRIVE_ROOT = "/content/drive/MyDrive/vggt-omega-aura-benchmark"   # where predictions, ground truth and results live.
# Work already saved there is skipped. To run EVERYTHING again from the images up, name an empty folder here,
# the same one in every notebook of the run. The Hugging Face token is still found in the usual folder's .env.
RUN_TAG = "phase7_front_medium"       # the folder of this run in persistent storage. A name from the time of the work:
                                      # every notebook of the run must use the same one, the results live under it
MODELS = ["vggt_omega_512", "vggt_1b"]
FINAL_SPLIT = "test"                  # never looked at while the rules were made
DEVELOPMENT_BLOCKS = [("val", 11)]    # the rules were tuned on this block, so it is left out of every comparison

In [ ]:
# === CODE SYNC (auto-generated by `python -m vggt_aura.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m vggt_aura.sync   and reopen this notebook.")

In [3]:
# --- 3. Start the session ---
from vggt_aura.session import start_session

session = start_session(persist_mode=PERSIST_MODE, drive_root=DRIVE_ROOT, require_gpu=False)

GPU: none (fine for download and inspection)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
persist root: /content/drive/MyDrive/vggt-omega-aura-benchmark
data root   : /content/data/fzi-aura (runtime disk, wiped at session end)


In [4]:
# --- 4. Load the run, keep the test scenes apart, build their tables ---
import pandas as pd
from vggt_aura import evaluation as ev, metrics as mt, pipeline as pl

pd.set_option("display.width", 260)
pd.set_option("display.max_columns", 40)
everything, runs = {}, {}
for model in MODELS:
    rows, scenes = pl.load_run(session.persist_root, RUN_TAG, model)
    development = scenes[[(s, b) in DEVELOPMENT_BLOCKS for s, b in zip(scenes["split"], scenes["block"])]]["scene_id"]
    rows, scenes = rows[~rows["scene_id"].isin(development)], scenes[~scenes["scene_id"].isin(development)]
    everything[model] = (rows.reset_index(drop=True), scenes.reset_index(drop=True))
    final = scenes[scenes["split"] == FINAL_SPLIT]
    runs[model] = (rows[rows["scene_id"].isin(final["scene_id"])].reset_index(drop=True), final.reset_index(drop=True))
    print(f"{model}: {len(final)} {FINAL_SPLIT} scenes in {final['block'].nunique()} blocks | {len(scenes) - len(final)} other scenes")
out_dir = session.persist_root / "metrics" / RUN_TAG / f"report_{FINAL_SPLIT}"
out_dir.mkdir(parents=True, exist_ok=True)


def show(title, table, digits=4):
    print()
    print("=====", title, "=====")
    print(table.round(digits).to_string(index=False) if len(table) else "(no rows)")


for model, (rows, scenes) in runs.items():
    if rows.empty:
        print(model, f": no {FINAL_SPLIT} results yet. Run `05_test_predict_gpu`, then `06_test_score`.")
        continue
    print()
    print("#" * 30, model, "|", FINAL_SPLIT, "|", scenes["scene_id"].nunique(), "scenes")
    report = ev.build_report(rows, scenes)
    for name, table in report.items():
        show(name, table)
        table.to_csv(out_dir / f"{model}_{name}.csv", index=False)

vggt_omega_512: 107 test scenes in 6 blocks | 287 other scenes
vggt_1b: 107 test scenes in 6 blocks | 287 other scenes

############################## vggt_omega_512 | test | 107 scenes

===== headline_by_protocol =====
         protocol  n_scenes    pixels  abs_rel  ci_low  ci_high  median  thin  delta125
      frame_scale       107 192174828   0.0733  0.0683   0.0787  0.0673 False    0.9468
frame_scale_shift       107 192174828   0.0765  0.0712   0.0818  0.0715 False    0.9448
       pose_scale       103 185346360   0.0862  0.0801   0.0929  0.0788 False    0.9418
   sequence_scale       107 192174828   0.0815  0.0765   0.0869  0.0762 False    0.9434
        unaligned       107 192174828   0.9800  0.9769   0.9827  0.9854 False    0.0000

===== motion =====
         stratum  n_scenes    pixels  abs_rel  ci_low  ci_high  median  thin  delta125
      background       107 174388863   0.0769  0.0722   0.0820  0.0724 False    0.9434
   moving object       101   5538260   0.2029  0.1783   0.

In [5]:
# --- 4b. Like for like: the test scenes against all other scenes, inside the same weather and light ---
# "difference" = test minus others. "intervals_overlap" True = the two are not clearly different.
for model, (rows, scenes) in everything.items():
    if (scenes["split"] == FINAL_SPLIT).any():
        table = ev.compare_splits(rows, scenes, FINAL_SPLIT)
        show(f"{model}: {FINAL_SPLIT} against train+val, per condition  [{mt.PRIMARY_PROTOCOL}]", table)
        table.to_csv(out_dir / f"{model}_{FINAL_SPLIT}_against_others.csv", index=False)
        for name, part in ((FINAL_SPLIT, scenes["split"] == FINAL_SPLIT), ("train+val", scenes["split"] != FINAL_SPLIT)):
            ids = scenes.loc[part, "scene_id"]
            moving = ev.paired_difference(rows[rows["scene_id"].isin(ids)], "motion_name", "moving object", "background")
            print(f"  {name:10} moving minus background: {moving['mean_difference']:+.4f} "
                  f"({moving['ci_low']:+.4f} to {moving['ci_high']:+.4f}), {moving['n_scenes']} scenes")


===== vggt_omega_512: test against train+val, per condition  [sequence_scale] =====
weather_group lighting  n_scenes_test  n_recordings_test  abs_rel_test  ci_low_test  ci_high_test  median_test  n_scenes_others  n_recordings_others  abs_rel_others  ci_low_others  ci_high_others  median_others  difference  intervals_overlap
          dry      day           86.0               12.0        0.0835       0.0780        0.0894       0.0770              148                   35          0.0806         0.0760          0.0856         0.0747      0.0029               True
          dry    night            NaN                NaN           NaN          NaN           NaN          NaN                2                    1          0.1213         0.1191          0.1234         0.1213         NaN               True
          wet      day           21.0                4.0        0.0734       0.0632        0.0864       0.0695               61                    9          0.0908         0.0826          

In [6]:
# --- 5. Model against model, paired by scene ---
if all(not runs[m][0].empty for m in MODELS) and len(MODELS) == 2:
    a, b = MODELS
    comparison = ev.compare_models(runs[a][0], runs[a][1], runs[b][0], runs[b][1], a, b)
    show(f"{a} minus {b}, paired by scene  [{mt.PRIMARY_PROTOCOL}]", comparison)
    comparison.to_csv(out_dir / "model_comparison.csv", index=False)
else:
    print("need results for exactly two models to compare")


===== vggt_omega_512 minus vggt_1b, paired by scene  [sequence_scale] =====
                   quantity  n_scenes  vggt_omega_512  vggt_1b  difference_a_minus_b  ci_low  ci_high  scenes_where_a_is_lower
         AbsRel, all pixels       107          0.0815   0.1251               -0.0436 -0.0503  -0.0375                      103
         AbsRel, background       107          0.0769   0.1165               -0.0396 -0.0452  -0.0339                      104
      AbsRel, parked object        93          0.0990   0.1471               -0.0482 -0.0656  -0.0344                       80
      AbsRel, moving object       101          0.2029   0.3691               -0.1661 -0.2155  -0.1200                       84
        rotation_deg_median        99          0.5911   2.4418               -1.8507 -3.6318  -0.6588                       92
     translation_deg_median        99          0.8212   2.2487               -1.4275 -2.4636  -0.6170                       60
                      auc30       

In [7]:
# --- 6. One line per scene, and the scenes where each model does worst ---
# "recording" is the drive a scene was cut from: many bad scenes from ONE recording are one problem, not many.
# scale_disagreement: 0 = pose and depth agree on the scale, 0.69 = they differ by a factor of two.
KEEP = ["scene_id", "block", "road_type", "weather_group", "lighting", "speed_kph_median", "abs_rel", "abs_rel_moving",
        "rotation_deg_median", "translation_deg_median", "ate_scale_only_pct_of_path", "pose_scale_over_depth_scale"]
for model, (rows, scenes) in runs.items():
    if rows.empty:
        continue
    overview = ev.scene_overview(rows, scenes)
    overview.to_csv(out_dir / f"{model}_scene_overview.csv", index=False)
    print()
    print("#" * 30, model)
    for title, by, moving_only in (("worst depth (AbsRel, all pixels)", "abs_rel", False),
                                   ("worst translation direction, moving scenes only", "translation_deg_median", True),
                                   ("pose scale and depth scale disagree most, moving scenes only", "scale_disagreement", True)):
        worst = ev.worst_scenes(overview, by, n=12, moving_only=moving_only)
        show(f"{title} | {worst['recording'].nunique()} recording(s) among these {len(worst)}",
             worst[[c for c in KEEP if c in worst]], digits=3)


############################## vggt_omega_512

===== worst depth (AbsRel, all pixels) | 7 recording(s) among these 12 =====
               scene_id  block road_type weather_group lighting  speed_kph_median  abs_rel  abs_rel_moving  rotation_deg_median  translation_deg_median  ate_scale_only_pct_of_path  pose_scale_over_depth_scale
 2025-08-04-11-19-58|54      2     urban           dry      day             4.645    0.170           0.362                0.215                   0.836                       1.172                        1.005
2026-06-02-17-05-20|111     11     urban           wet      day            48.330    0.168           0.209                0.698                   0.625                       0.338                        0.990
 2025-08-04-11-19-58|52      3     urban           dry      day             4.775    0.163           0.447                0.275                   0.697                       1.053                        0.931
 2025-08-04-11-19-58|43      2     urba